In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from firedrake import *
import matplotlib.pyplot as plt

import numpy as np

In [ ]:
# from Firedrake Matrix object (output of assemble of bilinear form) to numpy array
def mat2numpy(A):
  return A.petscmat.getValues(range(0, A.petscmat.getSize()[0]), range(0,  A.petscmat.getSize()[1]))
# from Firedrake Cofunction object (output of assemble of linear form) to numpy array
def dofvec2numpy(b):
  return b.dat.data

# Linear solvers in Firedrake / PETSc

We consider the diffusion-reaction problem

\begin{equation*}
\begin{cases}
- \Delta u + u = f & {\rm in} \ \Omega=(0,1)^2, \\
u = 0 & {\rm on} \ \partial\Omega,
\end{cases}
\end{equation*}
with $f(x,y)=(1+2\pi^2)\sin(\pi x)\sin(\pi y)$ and exact solution $u_\text{ex}(x,y)=\sin(\pi x)\sin(\pi y)$.

In [ ]:
N = 10
# Build the mesh
mesh = UnitSquareMesh(N, N, diagonal='left')

# Plot it
fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend()
ax.axis('equal')

# Define a discrete function space
V = FunctionSpace(mesh, 'P', 1)

# Define trial and test functions as belonging to the space
u = TrialFunction(V)
v = TestFunction(V)

# Data
x = SpatialCoordinate(mesh)
u_ex = Function(FunctionSpace(mesh, 'P', 2))    # higher-order FE space to avoid spoiling from interpolation error
u_ex.interpolate(sin(pi*x[0])*sin(pi*x[1]))
f = (1+2*pi*pi)*sin(pi*x[0])*sin(pi*x[1])
bcs = [ DirichletBC(V, Constant(0.0), (1,2,3,4)) ]

# Define the variational problem: bilinear form and rhs
a = dot(grad(u), grad(v)) * dx + u*v*dx
L = f*v*dx

In [ ]:
uh = Function(V)

# Define structures to assemble matrix and vector (does not actually perform the assembly: this is done at the first call of a solver)
vpb = LinearVariationalProblem(a, L, uh, bcs=bcs)

In [ ]:
from time import perf_counter

default_solver =  LinearVariationalSolver(vpb)
# NOT NECESSARILY THE ONE REPORTED HERE:
# https://www.firedrakeproject.org/solving-interface.html#default-solver-options

parameters = {'ksp_type': 'preonly', "pc_type": "lu",
                             "pc_factor_mat_solver_type": "mumps"}
direct_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters = {'ksp_type': 'gmres', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
gmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters = {'ksp_type': 'cg', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
cg_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
default_solver.solve()
print('DEFAULT SOLVER    -   elapsed time =', perf_counter() - t0, 's    -    # iter =', default_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
direct_solver.solve()
print('DIRECT SOLVER     -   elapsed time =', perf_counter() - t0, 's    -    # iter =', direct_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
gmres_solver.solve()
print('GMRES SOLVER      -   elapsed time =', perf_counter() - t0, 's    -    # iter =', gmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
cg_solver.solve()
print('CG SOLVER         -   elapsed time =', perf_counter() - t0, 's    -    # iter =', cg_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

In [ ]:
# Plot the numerical solution
fig, ax = plt.subplots()
q = tripcolor(uh, axes=ax)
fig.colorbar(q)

In [ ]:
# check default parameters
from firedrake.petsc import DEFAULT_KSP_PARAMETERS
print('default parameters:',DEFAULT_KSP_PARAMETERS,'\n')

# check stop criterion
A = mat2numpy(assemble(a, bcs=bcs))
b = dofvec2numpy(assemble(L, bcs=bcs))
u_array = dofvec2numpy(uh)
res = b - (A @ u_array)     # numpy matrix-product operator: @
print('CG SOLUTION  -   relative residual =', np.linalg.vector_norm(res)/np.linalg.vector_norm(b))

# further details on iterative solver
parameters.update({"ksp_monitor": None, 'ksp_monitor_true_residual': None})
cg_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)
uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
cg_solver.solve()

Preconditioning

In [ ]:
parameters = {'ksp_type': 'gmres', 'pc_type': 'none',
              'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}
gmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

parameters.update({'pc_type': 'ilu'})
pgmres_solver =  LinearVariationalSolver(vpb, solver_parameters=parameters)

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
gmres_solver.solve()
print('GMRES SOLVER      -   elapsed time =', perf_counter() - t0, 's    -    # iter =', gmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

uh.assign(0)    # initialization to avoid cross-effects from other solvers
t0 = perf_counter()
pgmres_solver.solve()
print('P-GMRES SOLVER    -   elapsed time =', perf_counter() - t0, 's    -    # iter =', pgmres_solver.snes.ksp.getIterationNumber())
print('                  -   error =', errornorm(u_ex, uh, 'H1'))

# Ex.1 - Monolithic Stokes system

### Cavity problem

\begin{equation*}
\begin{cases}
- \Delta \boldsymbol{u} + \nabla  p  = \boldsymbol{0} & {\rm in} \ \Omega=(0,1)^2, \\
{\rm div}\,\boldsymbol{u} = 0 & {\rm in} \ \Omega, \\
\boldsymbol{u} = \boldsymbol{g}_\text{D} & {\rm on} \ \Gamma_4.\\
\boldsymbol{u} = \boldsymbol{0} & {\rm on} \ \partial\Omega\setminus\Gamma_4, \\
\end{cases}
\end{equation*}

with $\boldsymbol{g}_\text{D} = 1\boldsymbol{i}$.

In [ ]:
# Build the mesh
n = 40
mesh = UnitSquareMesh(n, n)

fig, ax = plt.subplots()
triplot(mesh, axes=ax)
ax.legend()

In [ ]:
# Function spaces
V = VectorFunctionSpace(mesh, 'P', 2)
Q = FunctionSpace(mesh, 'P', 1)
W = MixedFunctionSpace([V, Q])
print('Ndofs - velocity :',V.dim(),', pressure :',Q.dim(),', total :',W.dim())

# Finite element functions
u, p = TrialFunctions(W)
v, q = TestFunctions(W)

In [ ]:
# Boundary conditions (strong)
bc4 = DirichletBC(W.sub(0), Constant((1., 0.)), 4)
bcRest = DirichletBC(W.sub(0), Constant((0., 0.)), [1,2,3])
bcs = (bc4, bcRest)

# Variational formulation
a = inner(grad(u), grad(v)) * dx - div(v) * p * dx + q * div(u) * dx
L = inner(Constant((0.0,0.0)), v) * dx
  # Dummy rhs (=0) to ensure that the solve recognize a==L as a linear problem

# Null space for fully Dirichlet conditions
nsp = MixedVectorSpaceBasis(
    W, [W.sub(0), VectorSpaceBasis(constant=True)]
)

# Solution (NB: do not use the same name u,v,p,q of the trial/test functions)
from time import perf_counter
wh = Function(W)
vpb = LinearVariationalProblem(a, L, wh, bcs)

In [ ]:
... SOLVE ...

In [ ]:
fig, ax = plt.subplots()
col = tripcolor(ph, axes=ax)
plt.colorbar(col)
plt.title('pressure')
fig, ax = plt.subplots()
#triplot(mesh, axes=ax)
col = quiver(uh, axes=ax)
plt.colorbar(col)
plt.title('velocity')